# Étape 4 - Apprentissage semi-supervisé

## Objectifs

- Construire un split `train / validation / test` strictement à partir des labels forts.
- Entraîner un baseline supervisé sur les labels forts uniquement.
- Entraîner un modèle sur les labels faibles produits par le clustering.
- Poursuivre l'entraînement de ce même modèle sur les labels forts.
- Comparer les performances finales entre apprentissage supervisé et apprentissage semi-supervisé.

## Principe méthodologique

- Le jeu de test est construit uniquement à partir des labels forts.
- Les labels faibles ne remplacent jamais les labels forts.
- Les pseudo-labels ne servent qu'à une phase de pré-entraînement, puis le modèle est affiné sur les labels forts.
- La comparaison finale s'effectue sur le même jeu de test jamais vu.

In [1]:
from __future__ import annotations

from pathlib import Path
import copy
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18

PROJECT_ROOT_CANDIDATE = Path.cwd().resolve()
for candidate in [PROJECT_ROOT_CANDIDATE, *PROJECT_ROOT_CANDIDATE.parents]:
    if (candidate / "src").exists() and (candidate / "environment.yml").exists():
        if str(candidate) not in sys.path:
            sys.path.append(str(candidate))
        break

from src.notebook_utils import (
    build_figure_saver,
    configure_notebook,
    ensure_directory,
    find_project_root,
    set_global_seed,
)

configure_notebook(display_max_columns=100, display_max_rows=100)


In [2]:
PROJECT_ROOT = find_project_root()
IMAGE_INDEX_PATH = PROJECT_ROOT / "data" / "interim" / "image_index.csv"
WEAK_LABEL_PATH = PROJECT_ROOT / "data" / "processed" / "clustering" / "resnet18_unlabeled_weak_labels_kmeans.csv"
OUTPUT_DIR = ensure_directory(PROJECT_ROOT / "data" / "processed" / "semi_supervised")
FIGURES_DIR = ensure_directory(PROJECT_ROOT / "reports" / "figures" / "semi_supervised")

RANDOM_SEED = 42
set_global_seed(RANDOM_SEED, torch_module=torch)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

STRONG_TRAIN_FRACTION = 0.60
STRONG_VALIDATION_FRACTION = 0.20
STRONG_TEST_FRACTION = 0.20

STRONG_TRAIN_BATCH_SIZE = 16
WEAK_TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
NUM_WORKERS = 0

SUPERVISED_EPOCHS = 8
WEAK_PRETRAIN_EPOCHS = 4
SEMISUPERVISED_FINETUNE_EPOCHS = 6

SUPERVISED_LR = 1e-3
WEAK_PRETRAIN_LR = 1e-3
SEMISUPERVISED_FINETUNE_LR = 5e-4

SAVE_FIGURES = True
save_figure = build_figure_saver(FIGURES_DIR, enabled=SAVE_FIGURES)

LABEL_NAME_MAP = {0: "normal", 1: "cancer"}
WEIGHTS = ResNet18_Weights.DEFAULT

print(f"PROJECT_ROOT                  : {PROJECT_ROOT}")
print(f"IMAGE_INDEX_PATH              : {IMAGE_INDEX_PATH}")
print(f"WEAK_LABEL_PATH               : {WEAK_LABEL_PATH}")
print(f"OUTPUT_DIR                    : {OUTPUT_DIR}")
print(f"FIGURES_DIR                   : {FIGURES_DIR}")
print(f"DEVICE                        : {DEVICE}")
print(f"SUPERVISED_EPOCHS             : {SUPERVISED_EPOCHS}")
print(f"WEAK_PRETRAIN_EPOCHS          : {WEAK_PRETRAIN_EPOCHS}")
print(f"SEMISUPERVISED_FINETUNE_EPOCHS: {SEMISUPERVISED_FINETUNE_EPOCHS}")
if DEVICE.type == "cuda":
    print(f"CUDA device                   : {torch.cuda.get_device_name(0)}")


PROJECT_ROOT                  : /home/maxime/projects/brainscan-semisupervised
IMAGE_INDEX_PATH              : /home/maxime/projects/brainscan-semisupervised/data/interim/image_index.csv
WEAK_LABEL_PATH               : /home/maxime/projects/brainscan-semisupervised/data/processed/clustering/resnet18_unlabeled_weak_labels_kmeans.csv
OUTPUT_DIR                    : /home/maxime/projects/brainscan-semisupervised/data/processed/semi_supervised
FIGURES_DIR                   : /home/maxime/projects/brainscan-semisupervised/reports/figures/semi_supervised
DEVICE                        : cuda
SUPERVISED_EPOCHS             : 8
WEAK_PRETRAIN_EPOCHS          : 4
SEMISUPERVISED_FINETUNE_EPOCHS: 6
CUDA device                   : NVIDIA GeForce RTX 5060 Laptop GPU


In [3]:
class BrainScanClassificationDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, project_root: Path, transform: transforms.Compose):
        self.frame = frame.reset_index(drop=True).copy()
        self.project_root = project_root
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict[str, object]:
        row = self.frame.iloc[idx]
        image_path = self.project_root / row["relative_path"]

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image_tensor = self.transform(image)

        return {
            "image": image_tensor,
            "label": int(row["label"]),
            "relative_path": row["relative_path"],
            "label_name": row["label_name"],
            "dataset_role": row["dataset_role"],
        }


def build_transforms() -> tuple[transforms.Compose, transforms.Compose]:
    weights_transform = WEIGHTS.transforms()
    train_transform = transforms.Compose(
        [
            transforms.Resize((224, 224), antialias=True),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=weights_transform.mean, std=weights_transform.std),
        ]
    )
    eval_transform = transforms.Compose(
        [
            transforms.Resize((224, 224), antialias=True),
            transforms.ToTensor(),
            transforms.Normalize(mean=weights_transform.mean, std=weights_transform.std),
        ]
    )
    return train_transform, eval_transform


def make_loader(frame: pd.DataFrame, transform: transforms.Compose, *, batch_size: int, shuffle: bool) -> DataLoader:
    dataset = BrainScanClassificationDataset(frame, PROJECT_ROOT, transform)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


def build_resnet18_binary_classifier(seed: int = RANDOM_SEED) -> nn.Module:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = resnet18(weights=WEIGHTS)
    for parameter in model.parameters():
        parameter.requires_grad = False

    model.fc = nn.Linear(model.fc.in_features, 2)
    return model.to(DEVICE)


def compute_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
    }


@torch.inference_mode()
def predict_with_model(model: nn.Module, loader: DataLoader, split_name: str) -> tuple[pd.DataFrame, dict[str, float]]:
    model.eval()
    rows = []

    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=PIN_MEMORY)
        logits = model(images)
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()
        predicted_labels = logits.argmax(dim=1).cpu().numpy().astype(int)
        true_labels = batch["label"].cpu().numpy().astype(int)

        for idx in range(len(predicted_labels)):
            rows.append(
                {
                    "split_name": split_name,
                    "relative_path": batch["relative_path"][idx],
                    "true_label": int(true_labels[idx]),
                    "true_label_name": LABEL_NAME_MAP[int(true_labels[idx])],
                    "pred_label": int(predicted_labels[idx]),
                    "pred_label_name": LABEL_NAME_MAP[int(predicted_labels[idx])],
                    "prob_normal": float(probabilities[idx, 0]),
                    "prob_cancer": float(probabilities[idx, 1]),
                }
            )

    predictions_df = pd.DataFrame(rows)
    metrics = compute_binary_metrics(
        predictions_df["true_label"].to_numpy(dtype=int),
        predictions_df["pred_label"].to_numpy(dtype=int),
    )
    return predictions_df, metrics


def train_classifier(
    model: nn.Module,
    train_loader: DataLoader,
    validation_loader: DataLoader,
    *,
    stage_name: str,
    epochs: int,
    learning_rate: float,
) -> tuple[nn.Module, pd.DataFrame, pd.DataFrame, dict[str, float]]:
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=learning_rate)

    best_state = copy.deepcopy(model.state_dict())
    best_validation_f1 = -1.0
    history_rows = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        seen_samples = 0

        for batch in train_loader:
            images = batch["image"].to(DEVICE, non_blocking=PIN_MEMORY)
            labels = batch["label"].to(DEVICE)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            seen_samples += images.size(0)

        validation_predictions_df, validation_metrics = predict_with_model(model, validation_loader, split_name="validation")
        epoch_train_loss = running_loss / max(seen_samples, 1)

        history_rows.append(
            {
                "stage_name": stage_name,
                "epoch": epoch,
                "train_loss": epoch_train_loss,
                **validation_metrics,
            }
        )

        if validation_metrics["f1"] > best_validation_f1:
            best_validation_f1 = validation_metrics["f1"]
            best_state = copy.deepcopy(model.state_dict())
            best_validation_predictions_df = validation_predictions_df.copy()
            best_validation_metrics = validation_metrics.copy()

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history_rows)
    return model, history_df, best_validation_predictions_df, best_validation_metrics


## 1. Charger les labels forts et les labels faibles

On construit ici deux tables distinctes :

- un sous-ensemble **fortement labellisé** pour le split `train / validation / test` ;
- un sous-ensemble **faiblement labellisé** issu du clustering, utilisé uniquement pour le pré-entraînement semi-supervisé.

In [4]:
image_index_df = pd.read_csv(IMAGE_INDEX_PATH)
weak_label_df = pd.read_csv(WEAK_LABEL_PATH)

strong_df = image_index_df.loc[
    image_index_df["source_split"] == "strong_labeled_pool",
    ["relative_path", "source_split", "label_strong", "label_strong_name"],
].copy()
strong_df["label"] = strong_df["label_strong"].astype(int)
strong_df["label_name"] = strong_df["label"].map(LABEL_NAME_MAP)

weak_df = weak_label_df.merge(
    image_index_df[["relative_path", "source_split"]],
    on="relative_path",
    how="left",
    validate="one_to_one",
)
weak_df["label"] = weak_df["weak_label_kmeans"].astype(int)
weak_df["label_name"] = weak_df["label"].map(LABEL_NAME_MAP)

display(strong_df.head(3))
display(weak_df.head(3))


,relative_path,source_split,label_strong,label_strong_name,label,label_name
0,data/raw/avec_labels/cancer/05340cd4-3bb2-459d...,strong_labeled_pool,1.0,cancer,1,cancer
1,data/raw/avec_labels/cancer/0c6f3641-60d9-4a76...,strong_labeled_pool,1.0,cancer,1,cancer
2,data/raw/avec_labels/cancer/0f718241-8f63-4b55...,strong_labeled_pool,1.0,cancer,1,cancer


,relative_path,source_split_x,cluster_kmeans_pca10,weak_label_kmeans,weak_label_name_kmeans,source_split_y,label,label_name
0,data/raw/sans_label/001b158a-7af8-451e-bf31-3a...,unlabeled_pool,1,1,cancer,unlabeled_pool,1,cancer
1,data/raw/sans_label/00366e8d-5520-4d3c-a70b-91...,unlabeled_pool,1,1,cancer,unlabeled_pool,1,cancer
2,data/raw/sans_label/00455a62-f79f-4072-9a23-49...,unlabeled_pool,0,0,normal,unlabeled_pool,0,normal


In [5]:
strong_train_df, strong_temp_df = train_test_split(
    strong_df,
    test_size=STRONG_VALIDATION_FRACTION + STRONG_TEST_FRACTION,
    stratify=strong_df["label"],
    random_state=RANDOM_SEED,
)

strong_validation_df, strong_test_df = train_test_split(
    strong_temp_df,
    test_size=0.5,
    stratify=strong_temp_df["label"],
    random_state=RANDOM_SEED,
)

strong_train_df = strong_train_df.copy()
strong_validation_df = strong_validation_df.copy()
strong_test_df = strong_test_df.copy()
weak_df = weak_df.copy()

strong_train_df["dataset_role"] = "strong_train"
strong_validation_df["dataset_role"] = "strong_validation"
strong_test_df["dataset_role"] = "strong_test"
weak_df["dataset_role"] = "weak_train"

split_manifest_df = pd.concat(
    [
        strong_train_df[["relative_path", "dataset_role", "label_name"]],
        strong_validation_df[["relative_path", "dataset_role", "label_name"]],
        strong_test_df[["relative_path", "dataset_role", "label_name"]],
        weak_df[["relative_path", "dataset_role", "label_name"]],
    ],
    ignore_index=True,
)

split_summary_df = (
    split_manifest_df.groupby(["dataset_role", "label_name"])
    .size()
    .rename("n_images")
    .reset_index()
    .sort_values(["dataset_role", "label_name"])
)

display(split_summary_df)


,dataset_role,label_name,n_images
0,strong_test,cancer,10
1,strong_test,normal,10
2,strong_train,cancer,30
3,strong_train,normal,30
4,strong_validation,cancer,10
5,strong_validation,normal,10
6,weak_train,cancer,770
7,weak_train,normal,636


## 2. Préprocessing et DataLoaders

Une légère augmentation est appliquée sur les jeux d'entraînement, tandis que les jeux de validation et de test utilisent un preprocessing déterministe.

In [6]:
train_transform, eval_transform = build_transforms()

strong_train_loader = make_loader(strong_train_df, train_transform, batch_size=STRONG_TRAIN_BATCH_SIZE, shuffle=True)
strong_validation_loader = make_loader(strong_validation_df, eval_transform, batch_size=EVAL_BATCH_SIZE, shuffle=False)
strong_test_loader = make_loader(strong_test_df, eval_transform, batch_size=EVAL_BATCH_SIZE, shuffle=False)
weak_train_loader = make_loader(weak_df, train_transform, batch_size=WEAK_TRAIN_BATCH_SIZE, shuffle=True)

loader_summary_df = pd.DataFrame(
    {
        "loader_name": [
            "strong_train_loader",
            "strong_validation_loader",
            "strong_test_loader",
            "weak_train_loader",
        ],
        "n_images": [
            len(strong_train_df),
            len(strong_validation_df),
            len(strong_test_df),
            len(weak_df),
        ],
        "batch_size": [
            STRONG_TRAIN_BATCH_SIZE,
            EVAL_BATCH_SIZE,
            EVAL_BATCH_SIZE,
            WEAK_TRAIN_BATCH_SIZE,
        ],
    }
)

display(loader_summary_df)


,loader_name,n_images,batch_size
0,strong_train_loader,60,16
1,strong_validation_loader,20,32
2,strong_test_loader,20,32
3,weak_train_loader,1406,32


## 3. Expérience 1 - Baseline supervisée

Le premier modèle est entraîné uniquement sur les labels forts du split d'entraînement.

In [7]:
supervised_model = build_resnet18_binary_classifier(seed=RANDOM_SEED)
supervised_start = time.perf_counter()
supervised_model, supervised_history_df, supervised_validation_predictions_df, supervised_validation_metrics = train_classifier(
    supervised_model,
    strong_train_loader,
    strong_validation_loader,
    stage_name="supervised",
    epochs=SUPERVISED_EPOCHS,
    learning_rate=SUPERVISED_LR,
)
supervised_training_seconds = time.perf_counter() - supervised_start

supervised_test_predictions_df, supervised_test_metrics = predict_with_model(
    supervised_model,
    strong_test_loader,
    split_name="test",
)

display(supervised_history_df)
display(pd.DataFrame([supervised_test_metrics]))


,stage_name,epoch,train_loss,accuracy,f1,precision,recall
0,supervised,1,0.812218,0.55,0.640000,0.533333,0.8
1,supervised,2,0.588855,0.65,0.740741,0.588235,1.0
2,supervised,3,0.497413,0.65,0.720000,0.600000,0.9
3,supervised,4,0.379798,0.65,0.695652,0.615385,0.8
4,supervised,5,0.343853,0.70,0.750000,0.642857,0.9
5,supervised,6,0.335061,0.70,0.750000,0.642857,0.9
6,supervised,7,0.299802,0.70,0.750000,0.642857,0.9
7,supervised,8,0.271893,0.70,0.750000,0.642857,0.9


,accuracy,f1,precision,recall
0,0.8,0.8,0.8,0.8


## 4. Expérience 2 - Pré-entraînement sur labels faibles

Le second modèle commence par apprendre sur le jeu faiblement labellisé. La validation est suivie sur les labels forts, sans jamais toucher au jeu de test.

In [8]:
semi_supervised_model = build_resnet18_binary_classifier(seed=RANDOM_SEED)
weak_pretrain_start = time.perf_counter()
semi_supervised_model, weak_pretrain_history_df, weak_validation_predictions_df, weak_validation_metrics = train_classifier(
    semi_supervised_model,
    weak_train_loader,
    strong_validation_loader,
    stage_name="weak_pretrain",
    epochs=WEAK_PRETRAIN_EPOCHS,
    learning_rate=WEAK_PRETRAIN_LR,
)
weak_pretrain_seconds = time.perf_counter() - weak_pretrain_start

weak_only_test_predictions_df, weak_only_test_metrics = predict_with_model(
    semi_supervised_model,
    strong_test_loader,
    split_name="test",
)

display(weak_pretrain_history_df)
display(pd.DataFrame([weak_only_test_metrics]))


,stage_name,epoch,train_loss,accuracy,f1,precision,recall
0,weak_pretrain,1,0.356753,0.80,0.777778,0.875,0.7
1,weak_pretrain,2,0.176840,0.80,0.750000,1.000,0.6
2,weak_pretrain,3,0.130342,0.75,0.666667,1.000,0.5
3,weak_pretrain,4,0.124649,0.75,0.666667,1.000,0.5


,accuracy,f1,precision,recall
0,0.85,0.823529,1.0,0.7


## 5. Expérience 3 - Fine-tuning sur labels forts

On poursuit l'entraînement du modèle pré-entraîné sur labels faibles avec le jeu d'entraînement fortement labellisé.

In [9]:
finetune_start = time.perf_counter()
semi_supervised_model, semi_finetune_history_df, semi_validation_predictions_df, semi_validation_metrics = train_classifier(
    semi_supervised_model,
    strong_train_loader,
    strong_validation_loader,
    stage_name="semi_supervised_finetune",
    epochs=SEMISUPERVISED_FINETUNE_EPOCHS,
    learning_rate=SEMISUPERVISED_FINETUNE_LR,
)
semi_finetune_seconds = time.perf_counter() - finetune_start

semi_supervised_test_predictions_df, semi_supervised_test_metrics = predict_with_model(
    semi_supervised_model,
    strong_test_loader,
    split_name="test",
)

display(semi_finetune_history_df)
display(pd.DataFrame([semi_supervised_test_metrics]))


,stage_name,epoch,train_loss,accuracy,f1,precision,recall
0,semi_supervised_finetune,1,0.418779,0.8,0.75,1.0,0.6
1,semi_supervised_finetune,2,0.380343,0.8,0.75,1.0,0.6
2,semi_supervised_finetune,3,0.349678,0.8,0.75,1.0,0.6
3,semi_supervised_finetune,4,0.354030,0.8,0.75,1.0,0.6
4,semi_supervised_finetune,5,0.256846,0.8,0.75,1.0,0.6
5,semi_supervised_finetune,6,0.282997,0.8,0.75,1.0,0.6


,accuracy,f1,precision,recall
0,0.8,0.75,1.0,0.6


## 6. Comparaison des performances

On compare ici trois états utiles :

- le baseline supervisé ;
- le modèle après pré-entraînement sur labels faibles uniquement ;
- le modèle semi-supervisé final après fine-tuning sur labels forts.

In [10]:
experiment_comparison_df = pd.DataFrame(
    [
        {
            "experiment": "supervised_only",
            **supervised_test_metrics,
            "training_seconds": round(supervised_training_seconds, 2),
        },
        {
            "experiment": "weak_pretrain_only",
            **weak_only_test_metrics,
            "training_seconds": round(weak_pretrain_seconds, 2),
        },
        {
            "experiment": "semi_supervised_final",
            **semi_supervised_test_metrics,
            "training_seconds": round(weak_pretrain_seconds + semi_finetune_seconds, 2),
        },
    ]
).sort_values("f1", ascending=False).reset_index(drop=True)

display(experiment_comparison_df)


,experiment,accuracy,f1,precision,recall,training_seconds
0,weak_pretrain_only,0.85,0.823529,1.0,0.7,22.83
1,supervised_only,0.80,0.800000,0.8,0.8,3.30
2,semi_supervised_final,0.80,0.750000,1.0,0.6,24.67


In [11]:
history_comparison_df = pd.concat(
    [supervised_history_df, weak_pretrain_history_df, semi_finetune_history_df],
    ignore_index=True,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(data=history_comparison_df, x="epoch", y="train_loss", hue="stage_name", marker="o", ax=axes[0])
axes[0].set_title("Perte d'entraînement par phase")
axes[0].set_xlabel("Époque")
axes[0].set_ylabel("Train loss")

sns.lineplot(data=history_comparison_df, x="epoch", y="f1", hue="stage_name", marker="o", ax=axes[1])
axes[1].set_title("F1 de validation par phase")
axes[1].set_xlabel("Époque")
axes[1].set_ylabel("Validation F1")

plt.tight_layout()
save_figure(fig, "04_semi_supervised_training_histories")
plt.show()


Figure sauvegardée : /home/maxime/projects/brainscan-semisupervised/reports/figures/semi_supervised/04_semi_supervised_training_histories.png


/tmp/ipykernel_740/2534735297.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
def build_confusion_table(predictions_df: pd.DataFrame) -> pd.DataFrame:
    matrix = confusion_matrix(predictions_df["true_label"], predictions_df["pred_label"], labels=[0, 1])
    return pd.DataFrame(matrix, index=["true_normal", "true_cancer"], columns=["pred_normal", "pred_cancer"])


supervised_confusion_df = build_confusion_table(supervised_test_predictions_df)
weak_only_confusion_df = build_confusion_table(weak_only_test_predictions_df)
semi_supervised_confusion_df = build_confusion_table(semi_supervised_test_predictions_df)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(supervised_confusion_df, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Supervisé uniquement")

sns.heatmap(weak_only_confusion_df, annot=True, fmt="d", cmap="Oranges", ax=axes[1])
axes[1].set_title("Pré-entraînement faible uniquement")

sns.heatmap(semi_supervised_confusion_df, annot=True, fmt="d", cmap="Greens", ax=axes[2])
axes[2].set_title("Semi-supervisé final")

plt.tight_layout()
save_figure(fig, "04_semi_supervised_confusion_matrices")
plt.show()

display(supervised_confusion_df)
display(weak_only_confusion_df)
display(semi_supervised_confusion_df)


Figure sauvegardée : /home/maxime/projects/brainscan-semisupervised/reports/figures/semi_supervised/04_semi_supervised_confusion_matrices.png


/tmp/ipykernel_740/1046201992.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,pred_normal,pred_cancer
true_normal,8,2
true_cancer,2,8


,pred_normal,pred_cancer
true_normal,10,0
true_cancer,3,7


,pred_normal,pred_cancer
true_normal,10,0
true_cancer,4,6


## 7. Export des artefacts

On sauvegarde les métriques, l'historique et les prédictions du jeu de test pour documenter proprement la comparaison.

In [13]:
comparison_path = OUTPUT_DIR / "semi_supervised_experiment_comparison.csv"
history_path = OUTPUT_DIR / "semi_supervised_training_history.csv"
predictions_path = OUTPUT_DIR / "semi_supervised_test_predictions.csv"
split_manifest_path = OUTPUT_DIR / "semi_supervised_split_manifest.csv"
run_config_path = OUTPUT_DIR / "semi_supervised_run_config.json"

combined_test_predictions_df = pd.concat(
    [
        supervised_test_predictions_df.assign(experiment="supervised_only"),
        weak_only_test_predictions_df.assign(experiment="weak_pretrain_only"),
        semi_supervised_test_predictions_df.assign(experiment="semi_supervised_final"),
    ],
    ignore_index=True,
)

run_config = {
    "random_seed": RANDOM_SEED,
    "device": str(DEVICE),
    "strong_train_fraction": STRONG_TRAIN_FRACTION,
    "strong_validation_fraction": STRONG_VALIDATION_FRACTION,
    "strong_test_fraction": STRONG_TEST_FRACTION,
    "supervised_epochs": SUPERVISED_EPOCHS,
    "weak_pretrain_epochs": WEAK_PRETRAIN_EPOCHS,
    "semi_supervised_finetune_epochs": SEMISUPERVISED_FINETUNE_EPOCHS,
    "supervised_learning_rate": SUPERVISED_LR,
    "weak_pretrain_learning_rate": WEAK_PRETRAIN_LR,
    "semi_supervised_finetune_learning_rate": SEMISUPERVISED_FINETUNE_LR,
    "strong_train_batch_size": STRONG_TRAIN_BATCH_SIZE,
    "weak_train_batch_size": WEAK_TRAIN_BATCH_SIZE,
}

experiment_comparison_df.to_csv(comparison_path, index=False)
history_comparison_df.to_csv(history_path, index=False)
combined_test_predictions_df.to_csv(predictions_path, index=False)
split_manifest_df.to_csv(split_manifest_path, index=False)
run_config_path.write_text(json.dumps(run_config, indent=2, ensure_ascii=False))

artifact_df = pd.DataFrame(
    {
        "artifact": [
            "experiment_comparison_csv",
            "training_history_csv",
            "test_predictions_csv",
            "split_manifest_csv",
            "run_config_json",
        ],
        "path": [
            str(comparison_path),
            str(history_path),
            str(predictions_path),
            str(split_manifest_path),
            str(run_config_path),
        ],
    }
)

display(artifact_df)


,artifact,path
0,experiment_comparison_csv,/home/maxime/projects/brainscan-semisupervised...
1,training_history_csv,/home/maxime/projects/brainscan-semisupervised...
2,test_predictions_csv,/home/maxime/projects/brainscan-semisupervised...
3,split_manifest_csv,/home/maxime/projects/brainscan-semisupervised...
4,run_config_json,/home/maxime/projects/brainscan-semisupervised...


## 8. Observations et interprétation

### Résultat principal de cette exécution

Sur cette expérimentation, le modèle **semi-supervisé final** ne surpasse pas le baseline **supervisé uniquement** sur le jeu de test. Les métriques observées sont les suivantes :

- **supervisé uniquement** : `accuracy = 0,80`, `F1 = 0,80`, `precision = 0,80`, `recall = 0,80` ;
- **pré-entraînement sur labels faibles uniquement** : `accuracy = 0,85`, `F1 = 0,824`, `precision = 1,00`, `recall = 0,70` ;
- **semi-supervisé final après fine-tuning** : `accuracy = 0,80`, `F1 = 0,75`, `precision = 1,00`, `recall = 0,60`.

### Comment interpréter ces résultats

Le signal le plus intéressant est que le modèle pré-entraîné uniquement sur les labels faibles obtient déjà des performances non triviales sur le jeu de test fortement labellisé. Cela confirme que les pseudo-labels générés à l'étape 3 contiennent une information utile.

En revanche, dans cette configuration précise, le **fine-tuning sur le petit jeu fortement labellisé ne transforme pas ce signal en gain final de performance**. Au contraire, le rappel du modèle semi-supervisé final chute à `0,60`, ce qui est problématique dans un contexte de détection de tumeur, où l'on souhaite limiter au maximum les faux négatifs.

### Ce que l'on peut conclure à ce stade

- les labels faibles produits par le clustering ne sont pas inutiles ; ils apportent bien un signal exploitable ;
- en revanche, la stratégie de fine-tuning retenue ici n'est pas encore suffisante pour battre le baseline supervisé ;
- le pipeline semi-supervisé, dans sa forme actuelle, ne peut donc pas être retenu tel quel comme meilleure solution ;
- le baseline supervisé reste, pour cette expérience, la référence la plus robuste et la plus équilibrée.

### Hypothèses expliquant ce comportement

Plusieurs facteurs peuvent expliquer pourquoi le modèle semi-supervisé final n'améliore pas les performances :

- les pseudo-labels issus du clustering restent bruités ;
- le jeu fortement labellisé est très petit (`100` images), ce qui rend le fine-tuning instable ;
- le choix d'un entraînement limité à la tête de classification d'un `ResNet18` pré-entraîné peut ne pas suffire à exploiter pleinement le signal ;
- les hyperparamètres de pré-entraînement et de fine-tuning restent perfectibles.

### Décision méthodologique pour la suite

À ce stade, la conclusion raisonnable est la suivante : **la piste semi-supervisée reste intéressante, mais elle n'est pas encore validée expérimentalement dans cette version du pipeline**.

Pour aller plus loin, il serait pertinent de tester :

- un filtrage des pseudo-labels les plus incertains ;
- un autre modèle de clustering ou un autre type d'embeddings ;
- un fine-tuning plus progressif, avec dégel partiel du backbone ;
- des réglages d'entraînement plus adaptés au très faible volume de labels forts.

En l'état, pour le rapport, on peut donc dire que **le semi-supervisé est faisable techniquement, mais que son bénéfice n'est pas encore démontré sur ce dataset avec cette implémentation**.